## "Which monthly acquisition cohorts demonstrate the highest long-term loyalty, and in exactly which month do we see the steepest drop-off in engagement so we can time our retention offers?"

### Just saying 'Retention rate is 25%' means nothing. BUT: "Customers acquired in November have a 40% retention rate in Month 3, but customers acquired in January drop to 15% by Month 3," you have just identified a massive behavioral trend. It tells the business exactly when to send a win-back email (right before that steep drop-off month) and which acquisition months bring in the highest-quality customers.

1. The Pull: Import get_warehouse_connection function and pull the fct_orders table into a pandas DataFrame.

In [ ]:
import pandas as pd
from db_connections import connect

con = connect.get_warehouse_connection('../mean_mug_analytics/mean_mug.duckdb')

# 2. SQL query in a string
query = "SELECT * FROM fct_orders"

# 3. send the query to the tunnel, catch results as a DF
df_orders = con.execute(query).df()

# 4. print first 5 rows
display(df_orders.head())


Successfully connected to warehouse: ../mean_mug_analytics/mean_mug.duckdb


,order_id,customer_id,store_location_id,promo_id,order_date,order_time,payment_type_lower,total_amount,total_items_in_basket
0,1,1,1,<NA>,2024-01-26,09:21:00,app,7.50,2
1,2,1,2,2,2024-02-12,07:54:00,app,4.25,1
2,3,2,2,2,2024-02-08,09:22:00,app,16.00,4
3,4,2,1,<NA>,2024-03-26,17:06:00,card,4.25,1
4,5,3,2,<NA>,2024-03-12,13:19:00,cash,10.00,3


2. The Acquisition Month: Find the very first month each customer ever made a purchase. This assigns them to their "Cohort."

In [3]:
# a. order_date needs to be month to find the acquisition month
df_orders['order_date'] = pd.to_datetime(df_orders['order_date'])
df_orders['order_month'] = df_orders['order_date'].dt.to_period("M")
df_orders['order_month']

0       2024-01
1       2024-02
2       2024-02
3       2024-03
4       2024-03
         ...   
2819    2024-03
2820    2024-03
2821    2024-02
2822    2024-03
2823    2024-03
Name: order_month, Length: 2824, dtype: period[M]

In [4]:
# b. cohort_month, to find a customer's cohort -- very first month they ever made a purchase--. Reset index to turn back to a DF
df_orders['cohort_month'] = df_orders.groupby('customer_id')['order_month'].transform('min')
df_orders.head()

,order_id,customer_id,store_location_id,promo_id,order_date,order_time,payment_type_lower,total_amount,total_items_in_basket,order_month,cohort_month
0,1,1,1,<NA>,2024-01-26,09:21:00,app,7.50,2,2024-01,2024-01
1,2,1,2,2,2024-02-12,07:54:00,app,4.25,1,2024-02,2024-01
2,3,2,2,2,2024-02-08,09:22:00,app,16.00,4,2024-02,2024-02
3,4,2,1,<NA>,2024-03-26,17:06:00,card,4.25,1,2024-03,2024-02
4,5,3,2,<NA>,2024-03-12,13:19:00,cash,10.00,3,2024-03,2024-03


3. The Activity Month: Calculate the month of every subsequent purchase they make.
We need to know how many months have passed between a customer's first purchase and their current purchase.

### FORMULA IS = Years Difference * 12 + Months Difference OR (Order Year - Cohort Year) * 12 + (Order Month - Cohort Month)

because if the year diff is 2024 - 2023 = 1 year it means *12 = 12 months. For the month 11 months - 12 months = -1 month. Added up it will be 11 months later, so index 11.

In [5]:
year_diff = df_orders['order_month'].dt.year - df_orders['cohort_month'].dt.year
month_diff = df_orders['order_month'].dt.month - df_orders['cohort_month'].dt.month
df_orders['cohort_index'] = year_diff*12 + month_diff
df_orders

,order_id,customer_id,store_location_id,promo_id,order_date,order_time,payment_type_lower,total_amount,total_items_in_basket,order_month,cohort_month,cohort_index
0,1,1,1,<NA>,2024-01-26,09:21:00,app,7.50,2,2024-01,2024-01,0
1,2,1,2,2,2024-02-12,07:54:00,app,4.25,1,2024-02,2024-01,1
2,3,2,2,2,2024-02-08,09:22:00,app,16.00,4,2024-02,2024-02,0
3,4,2,1,<NA>,2024-03-26,17:06:00,card,4.25,1,2024-03,2024-02,1
4,5,3,2,<NA>,2024-03-12,13:19:00,cash,10.00,3,2024-03,2024-03,0
...,...,...,...,...,...,...,...,...,...,...,...,...
2819,2821,405,1,2,2024-03-06,14:32:05,app,18.50,4,2024-03,2024-02,1
2820,2822,3,3,1,2024-03-25,09:49:02,card,20.50,3,2024-03,2024-03,0
2821,2823,381,2,1,2024-02-07,12:36:03,app,7.75,2,2024-02,2024-02,0
2822,2824,829,2,<NA>,2024-03-15,09:49:04,app,4.25,1,2024-03,2024-03,0


4. The Index (Cohort Age): Subtract the Acquisition Month from the Activity Month to get the "Cohort Index" (Month 0, Month 1, Month 2, etc.)

In [6]:
# a. see cohort month and cohort index group
cohort_data = df_orders.groupby(['cohort_month','cohort_index']).agg(total_count=('customer_id','nunique')).reset_index()
cohort_data

,cohort_month,cohort_index,total_count
0,2024-01,0,121
1,2024-01,1,80
2,2024-01,2,78
3,2024-02,0,276
4,2024-02,1,209
5,2024-03,0,462


5. The Matrix: Pivot the data so the rows are the Acquisition Cohorts, the columns are the Cohort Index, and the values are the count of unique customers.

In [7]:
# Build the matrix
cohort_matrix = cohort_data.pivot(index='cohort_month',columns='cohort_index',values='total_count')
cohort_matrix


cohort_index,0,1,2
cohort_month,,,
2024-01,121.0,80.0,78.0
2024-02,276.0,209.0,NaN
2024-03,462.0,NaN,NaN


6. The Percentages: Divide everything by the Month 0 starting count to get true retention percentages.

In [8]:
# Get the cohorts
cohort_sizes = cohort_matrix.iloc[:, 0]
# Divide the matrix by the cohorts for percentages, across rows meaning axis=0
retention_matrix = cohort_matrix.divide(cohort_sizes, axis=0)
retention_matrix.round(3)

cohort_index,0,1,2
cohort_month,,,
2024-01,1.0,0.661,0.645
2024-02,1.0,0.757,NaN
2024-03,1.0,NaN,NaN


according to the matrix february is better than january cohort, with 76% retention compared to 66% january. but there is not enough months to give a good analysis

### FINAL: Write back to DuckDB

In [9]:
# Add "Month_" to the front of every column name in the matrix
retention_matrix.columns = [f"Month_{col}" for col in retention_matrix.columns]

# flatten it
final_matrix = retention_matrix.reset_index()

# Convert the period data type to a standard string, otherwise "NotImplementedException: Not implemented Error: Data type 'period[M]' not recognized"
final_matrix['cohort_month'] = final_matrix['cohort_month'].astype(str)

In [ ]:
import importlib
from db_connections import bigquery_export 

# Force Jupyter to read the updated bigquery_export.py file
importlib.reload(bigquery_export)

# Updated function
bigquery_export.push_to_warehouse(final_matrix, "mart_cohort_retention")

Uploading mart_cohort_retention to BigQuery...
Success: mart_cohort_retention is live!


In [15]:
con.close()